In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import copy

# adatok előkészítése
data = np.load("20000.npy")
df = pd.DataFrame(data)
df["palya_id"] = df["eventID"].astype(str) + "_" + df["trackID"].astype(str)
primerek = df[df["parentID"] == 0]

parameterek = ['posX', 'posY', 'momDirX', 'momDirY', 'momDirZ', 'edep']
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 1000
BATCH_SIZE = 128

layers = [primerek[primerek["volumeID[3]"] == i].drop_duplicates(subset=["palya_id"]) for i in range(3)]


def prepare_layer_pair_data(layerA, layerB):
    # előkészíti, szűri és normalizálja két szomszédos réteg adatait
    common_labels = np.array(list(set(layerA["palya_id"]) & set(layerB["palya_id"])))
    np.random.shuffle(common_labels)

    split = int(0.9 * len(common_labels))
    train_labels, val_labels = common_labels[:split], common_labels[split:]

    l_train = [
        layerA[layerA["palya_id"].isin(train_labels)].set_index("palya_id").loc[train_labels],
        layerB[layerB["palya_id"].isin(train_labels)].set_index("palya_id").loc[train_labels]
    ]
    l_val = [
        layerA[layerA["palya_id"].isin(val_labels)].set_index("palya_id").loc[val_labels],
        layerB[layerB["palya_id"].isin(val_labels)].set_index("palya_id").loc[val_labels]
    ]

    # normalizálás az első réteg alapján
    x_mean = torch.tensor(l_train[0][parameterek].values.mean(axis=0), dtype=torch.float32)
    x_std = torch.tensor(l_train[0][parameterek].values.std(axis=0), dtype=torch.float32) + 1e-6

    return l_train, l_val, train_labels, val_labels, x_mean, x_std


# modell
class MatchHead(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.ql = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU())
        self.kl = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU())

    def forward(self, x_prev, x_next):
        q = self.ql(x_prev)
        k = self.kl(x_next)
        return q @ k.t()

class DualLayerMatcher(nn.Module):
    def __init__(self, input_dim, num_heads=3, hidden_dim=128):
        super().__init__()
        self.heads = nn.ModuleList([MatchHead(input_dim, hidden_dim) for _ in range(num_heads)])
        self.aggregator = nn.Linear(num_heads, 1)
        
    def match(self, x_prev, x_next, training=False):
        if training:
            x_prev = x_prev + 0.05 * torch.randn_like(x_prev) 
            
        dms = [head(x_prev, x_next) for head in self.heads] 
        stacked_dms = torch.stack(dms, dim=-1)
        final_dm = self.aggregator(stacked_dms).squeeze(-1)
        return torch.softmax(final_dm, dim=1)


# tanítás
def train_model_for_pair(pair_name, l_train, l_val, train_labels, val_labels, x_mean, x_std):
    print(f"\n{ pair_name }")
    
    model = DualLayerMatcher(len(parameterek), num_heads=3, hidden_dim=128).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

    for e in range(EPOCHS):
        model.train()
        np.random.shuffle(train_labels)
        
        train_accs = [] # Tanítási pontosság gyűjtése
        
        for i in range(0, len(train_labels), BATCH_SIZE):
            batch_labels = train_labels[i:i+BATCH_SIZE]
            if len(batch_labels) < 2: continue
                
            xs = []
            for df_lay in l_train:
                subset = df_lay.loc[batch_labels]
                X = (torch.tensor(subset[parameterek].values, dtype=torch.float32) - x_mean) / x_std
                xs.append(X.to(DEVICE))
                
            targets = torch.arange(len(batch_labels)).to(DEVICE)
            
            optimizer.zero_grad()
            probs = model.match(xs[0], xs[1], training=True)
            
            # tanítási batch pontosság kiszámítása
            train_accs.append((torch.argmax(probs, dim=1) == targets).float().mean().item() * 100)
            
            loss = nn.functional.nll_loss(torch.log(probs + 1e-8), targets)
            loss.backward()
            optimizer.step()
            
        scheduler.step()
        mean_train_acc = np.mean(train_accs) # Epoch átlag

        if e % 100 == 0:
            model.eval()
            val_accuracies = []
            with torch.no_grad():
                for i in range(0, len(val_labels), 512):
                    v_batch_labels = val_labels[i:i+512]
                    if len(v_batch_labels) < 2: continue
                    
                    v_xs = []
                    for df_lay in l_val:
                        subset = df_lay.loc[v_batch_labels]
                        v_X = (torch.tensor(subset[parameterek].values, dtype=torch.float32) - x_mean) / x_std
                        v_xs.append(v_X.to(DEVICE))
                    
                    v_targets = torch.arange(len(v_batch_labels)).to(DEVICE)
                    acc = (torch.argmax(model.match(v_xs[0], v_xs[1]), dim=1) == v_targets).float().mean().item() * 100
                    val_accuracies.append(acc)
                    
                mean_val_acc = np.mean(val_accuracies)
                
                # print Train % és Val %
                print(f"Epoch {e:4d} | Train Acc: {mean_train_acc:5.1f}% | Val Acc: {mean_val_acc:5.1f}%")

    return model


# L1 -> L2 (layers[0] és layers[1])
l1_train, l1_val, t_labels_1, v_labels_1, mean_1, std_1 = prepare_layer_pair_data(layers[0], layers[1])
model_L1_L2 = train_model_for_pair("L1-L2 Modell", l1_train, l1_val, t_labels_1, v_labels_1, mean_1, std_1)

# L2 -> L3 (layers[1] és layers[2])
l2_train, l2_val, t_labels_2, v_labels_2, mean_2, std_2 = prepare_layer_pair_data(layers[1], layers[2])
model_L2_L3 = train_model_for_pair("L2-L3 Modell", l2_train, l2_val, t_labels_2, v_labels_2, mean_2, std_2)